# Chapter 1

### Bayesian Inference

Probability: Statement of uncertainty. 2 rules
1. Sum rule: Probability of A or B (independent events)
    - Probability of rolling 2 or 4 with a die. 
    - P(2 or 4) = 1/6 + 1/6 = 0.33333... = 33.3%
1. Product rule: Probability of A and B (independent events)
    - Probability of rolling 2 and then 4 with a die
    - P(2 and 4) = 1/6 * 1/6 = 0.02777... = 2.8%

Credible intervals: Such an interval that the probability that the parameter falls inside it is x%. The wider the interval, the more uncertainty it incorporates. parameter is random, credible interval is fixed (Opposite to confidence interval where parameter is fixed and confidence interval is random)

Probability distribution: A distribution of a random variable specifies what values this variable can take, and with what probabilities. 2 types:
- discrete probability distribution: finite set of possible values.
- continuous probability distribution : infinitely many possible values. (can be visualized on a density plot)

Bayesian inference means updating one's belief about something as the new information becomes available. eg: Updating belief on parameters of a statistical model based on incoming data. It is different from classical approach:
1. Probability means proportion of outcome in classical approach. However, according to bayesian inference probability means degree of belief
2. In classical approach, parameters of statistical models are almost fixed. However, according to bayesian inference parameters are random variables (parameters have probability distribution and can take many different values for different probabilities)

Bayesian analysis is costly:
- Bayesian models account for uncertainty. eg: For a distribution of many equally likely outcome, the uncertainty is large
- Expert opinion can be integrated by statistical sound way (since bayesian analysis is about degree of belief), and thus incorporate uncertainty
- Does not rely on fixed p-values
- statistically strong with little data
- offers more flexibility for custom models and sometimes produces same result as classical approach

Conditional Probability: Probability of some event occurring, given that some other event has occurred. P(A | B)
- You have 2 orange ball and 1 blue ball in a box. what is the probability of picking 2 balls ?
- First Draw:
    - P(orange) = 2/3 → unconditional 
    - P(blue) = 1/3 → unconditional
- Second Draw: You put away the ball of the first draw. now you have 2 balls left in the box
    - After picking orange, the probability to pick blue, P(blue | orange) = 1/2 → conditional
    - After picking blue, the probability to pick orange, P(orange | blue) = 1 → conditional
- Using bayes theorem, the formula is: `p_A_given_B = p_B_given_A * p_A / p_B`

```
# All values are considered binary (success is 1 and failure is 0) in dataset
# Unconditional probability of an accident
p_accident = road_conditions["accident"].mean() # 0.0625

# Unconditional probability of the road being slippery
p_slippery = road_conditions["slippery"].mean() # 0.0892

# Probability of the road being slippery given there is an accident
p_slippery_given_accident = road_conditions.loc[road_conditions["accident"]]["slippery"].mean() # 0.7142

# Probability of an accident given the road is slippery
p_accident_given_slippery = p_slippery_given_accident * p_accident / p_slippery # 0.5
```

### Probability Distribution

- Binomial distribution: 
    - A discrete distribution 
    - two values: success (1) and failure(0)
    - One parameter: probability of success
    - `np.random.binomial(no_of_trials, success_rate, size = no_of_repeat_for_whole_process)`
    - Best practice: use `no_of_trials` as `1` and use `no_of_repeat_for_whole_process` as no of trials to get the trial values in array
    

# Chapter 2

### Bayes theorem on data

<center><img src="images/02.01.png"  style="width: 400px, height: 300px;"/></center>


- P(parameters) → prior distribution : what we know about the parameters before seeing any data (use uniform distribution for unknown or other posterior for known)
- P(parameters | data) → posterior distribution: : what we know about the parameters after having seen the data (Prior choice can impact posterior results- especially with little data)
- P(data | parameters) → data (possible likelyhood) according to our statistical model
- P(data) → scaling factor
- NOTE: Some priors, multiplied with specific likelihoods, yield known posteriors. These are called conjugate priors.

### Distribution and Probability functions

- Normal distribution : The normal distribution describes a continuous random variable with a symmetric, bell-shaped curve.
- Binomial distribution : The binomial distribution describes the number of successes in a fixed number of independent Bernoulli trials (A random trial that has binary outcome, probability of success is constant for each trial, each trial is independent of other trial).
- Uniform distribution: The uniform distribution describes an equal probability for all values in a given range.
- Poisson distribution: The Poisson distribution describes the number of events occurring within a fixed interval of time or space.
- Exponential Distribution: The exponential distribution describes the time between events in a Poisson process.
- Beta distribution : The beta distribution is a continuous probability distributions defined on the interval [0, 1], characterized by two shape parameters, 𝛼 and 𝛽, which determine the shape of the distribution. Used to model the distribution of probabilities and proportions. Can take various shapes (U-shaped, bell-shaped, etc.) depending on the two shape parameters.
- Gamma distribution : The gamma distribution is a  continuous probability distributions defined on the interval [0,∞),  characterized characterized by a shape parameter 𝑘 (also sometimes denoted as 𝛼) and a rate (or scale) parameter 𝜃 (sometimes denoted as 1/𝛽 ).  It extends the exponential distribution to model the waiting time until the k-th event in a Poisson process.  Used to model waiting times and is a generalization of the exponential distribution. Typically right-skewed and describes the waiting time until a certain number of events occur.
- Probability Density Function (PDF) : specify the probability of a continuous random variable falling within a particular range of values.
- Cumulative Distribution Function (CDF) : gives the probability that a random variable 𝑋 is less than or equal to a certain value 𝑥 . It is the integral of the PDF for continuous variables.
- Probability Mass Function (PMF) : is used for discrete random variables and gives the probability that a discrete random variable is exactly equal to some value.


```
# prior = P(parameters)
# posterior = P(parameters | data)
# scaling_factor = P(data)
# likelihood = P(data | parameters)
# Bayes formula: posterior = prior * likelihood / scaling_factor
# This method is used when posterior is not known (using grid approximation)
# If posterior is known, we can directly use a method to generate the specified distribution (eg: np.random.neta(2,4,100))
from scipy.stats import binom
from scipy.stats import uniform
num_heads = np.arange(0, 101, 1)
head_prob = np.arange(0, 1.01, 0.01)
coin = pd.DataFrame([(x, y) for x in num_heads for y in head_prob])
coin.columns = ["num_heads", "head_prob"]
coin["prior"] = uniform.pdf(coin["head_prob"]) # For unknown data we assume prob is equally likely for all events
coin["likelihood"] = binom.pmf(coin["num_heads"], 100, coin["head_prob"])
coin["posterior_prob"] = coin["prior"] * coin["likelihood"]
scaling_factor = coin["posterior_prob"].sum()
coin["posterior_prob"] = coin["posterior_prob"] / scaling_factor
# What's the probability of tossing heads with a coin, if we observed 75 heads in 100 tosses?
heads75 = coin.loc[coin["num_heads"] == 75]
scaling_factor = heads75["posterior_prob"].sum()
heads75["posterior_prob"] = heads75["posterior_prob"] / scaling_factor # Updated posterior
sns.lineplot(heads75["head_prob"], heads75["posterior_prob"])
plt.show()

# Announcing result in report
# With plot
sns.kdeplot(prior_draws, shade=True, label="prior")
sns.kdeplot(posterior_draws, shade=True, label="posterior")
# With point estimate
posterior_mean = np.mean(posterior_draws) 
posterior_median = np.median(posterior_draws)
posterior_p75 = np.percentile(posterior_draws,75)
# With credible interval (Confidence interval) as a measure of uncertainty in estimate
import arviz as az
hpd = az.hdi(posterior_draws, hdi_prob=0.9) # Highest Posterior Density (HPD)
print(hpd)
```

# Chapter 3

### A/B Testing

```
print(A_clicks) # [0 1 1 0 0 0 0 0 0 0 1 ... ]
print(B_clicks) # [0 0 0 1 0 0 0 1 1 0 1 ... ]

def simulate_beta_posterior(trials, beta_prior_a, beta_prior_b):
    num_successes = np.sum(trials)
    posterior_draws = np.random.beta( num_successes + beta_prior_a, len(trials) - num_successes + beta_prior_b, 10000 )
    return posterior_draws

A_posterior = simulate_beta_posterior(A_clicks, 1, 1)
B_posterior = simulate_beta_posterior(B_clicks, 1, 1)

sns.kdeplot(A_posterior, shade=True, label="A")
sns.kdeplot(B_posterior, shade=True, label="B")
plt.show()

# Posterior difference between B and A
diff = B_posterior - A_posterior
sns.kdeplot(diff, shade=True, label="difference: A-B")
plt.show()

# Probability of B being better
(diff > 0).mean()

# Probability of Expected (average) loss if our assumption is wrong
loss = diff[diff < 0]
expected_loss = loss.mean()

### Determining other factors from posterior analysis (eg: revenue)
# Specified constants
num_impressions = 1000
rev_per_click_A = 3.6
rev_per_click_B = 3

# Compute number of clicks
num_clicks_A = A_posterior * num_impressions
num_clicks_B = B_posterior * num_impressions

# Compute posterior revenue
rev_A = num_clicks_A * rev_per_click_A
rev_B = num_clicks_B * rev_per_click_B

import pymc3 as pm
# Visualize posterior
pm.plot_posterior(marketing_spending_draws, hdi_prob=0.95)
# Draw the forest plot
revenue = {"A": rev_A, "B": rev_B} # Collect posterior draws in a dictionary to draw multiple forest plots
pm.forestplot(revenue, hdi_prob=0.99)


# Regression
# Get point estimates of parameters
intercept_mean = intercept_draws.mean()
marketing_spending_mean = marketing_spending_draws.mean()
sd_mean = sd_draws.mean()
# Calculate mean of predictive distribution (How much sales can we expect if we spend $1000 on marketing?)
predictive_mean = intercept_mean + marketing_spending_mean * 1000   # y = c + m*x
# Simulate from predictive distribution using normal distribution
prediction_draws = np.random.normal(predictive_mean, sd_mean, size=10000)
```